## Colab Dependency Setup

Jalankan cell ini sekali di Colab atau kernel baru sebelum menjalankan notebook. `pandas` dipakai untuk dataframe/export CSV, sedangkan `pyarrow` hanya dibutuhkan jika `EXPORT_FORMAT = "parquet"`.


In [ ]:
%pip install -q pandas pyarrow


# YouTube Pilot Scraper for JudolLens

## tl;dr

Notebook ini mengumpulkan komentar YouTube berbasis **YouTube Data API v3** dengan pendekatan **seed video + keyword evidence**. Fokus tahap ini adalah membuat dataset pilot yang auditable dan siap anotasi, bukan auto-labeling final atau training model.

## Context & Methods

Strategi ini mengikuti temuan paper yang sudah dianalisis di `docs/research/judol-dataset-acquisition-analysis.md`:

- YouTube lebih kuat jika dimulai dari `seed_videos` atau `seed_channels`, bukan keyword-only scraping.
- Keyword hanya dipakai untuk `candidate_flag`, bukan untuk label final.
- Replies ikut dikumpulkan agar selaras dengan paper yang memakai top-level comments dan nested replies.
- Unicode dan obfuscation disimpan sebagai evidence tambahan untuk anotasi dan evaluasi robustnes.

### Key Assumptions

- Sumber data utama adalah **YouTube Data API v3**.
- Label default setiap komentar adalah `perlu_review`.
- `label_source` default adalah `unlabeled`.
- Identifier penulis tidak disimpan mentah kecuali memang dibutuhkan; default notebook ini memakai hash.


### API Key Notes for Colab

Jika memakai kernel Colab, `.env` lokal dari `D:\judol_detection` tidak otomatis terbaca karena runtime berjalan di cloud. Cara yang direkomendasikan adalah menambahkan secret bernama `YOUTUBE_API_KEY` di Colab Secrets. Notebook ini akan mencoba membaca urutannya dari environment/`.env`, lalu fallback ke Colab Secrets.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable
from urllib.parse import urlencode
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

import pandas as pd

pd.set_option("display.max_colwidth", 160)


## Setup

Atur parameter pilot di bawah ini. `SEED_VIDEOS` adalah jalur paling eksplisit dan direkomendasikan untuk putaran pertama. `SEED_CHANNELS` boleh dipakai jika ingin memperluas seed secara terkontrol.


In [ ]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "notebooks").exists() and (CURRENT_DIR / "docs").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.name == "notebooks") and (CURRENT_DIR.parent / "docs").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

ENV_PATH = PROJECT_ROOT / ".env"
PLACEHOLDER_API_KEYS = {"", "replace-with-your-youtube-data-api-key", "YOUR_API_KEY_HERE"}

def clean_api_key(value: str | None) -> str:
    candidate = (value or "").strip()
    return "" if candidate in PLACEHOLDER_API_KEYS else candidate

def read_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata
    except Exception:
        return ""

    try:
        return clean_api_key(userdata.get(name))
    except Exception:
        return ""

def load_simple_env(env_path: Path) -> dict[str, str]:
    if not env_path.exists():
        return {}

    loaded = {}
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            loaded[key] = value
    return loaded

LOADED_ENV = load_simple_env(ENV_PATH)
ENV_FILE_API_KEY = clean_api_key(LOADED_ENV.get("YOUTUBE_API_KEY"))
ENVIRONMENT_API_KEY = clean_api_key(os.getenv("YOUTUBE_API_KEY"))
COLAB_SECRET_API_KEY = ""

if ENV_FILE_API_KEY:
    YOUTUBE_API_KEY = ENV_FILE_API_KEY
    API_KEY_SOURCE = "env_file"
elif ENVIRONMENT_API_KEY:
    YOUTUBE_API_KEY = ENVIRONMENT_API_KEY
    API_KEY_SOURCE = "environment"
else:
    COLAB_SECRET_API_KEY = read_colab_secret("YOUTUBE_API_KEY")
    YOUTUBE_API_KEY = COLAB_SECRET_API_KEY
    API_KEY_SOURCE = "colab_secret" if YOUTUBE_API_KEY else ""

if YOUTUBE_API_KEY:
    os.environ["YOUTUBE_API_KEY"] = YOUTUBE_API_KEY

if not API_KEY_SOURCE and ENV_PATH.exists() and "YOUTUBE_API_KEY" in LOADED_ENV:
    API_KEY_SOURCE = "env_file_placeholder_or_empty"
elif not API_KEY_SOURCE:
    API_KEY_SOURCE = "not_found"

OUTPUT_DIR = PROJECT_ROOT / "data" / "interim"
RAW_OUTPUT_STEM = "youtube_comments_raw"
ANNOTATION_OUTPUT_STEM = "youtube_comments_annotation_queue"
EXPORT_FORMAT = "csv"  # "csv" or "parquet"

INCLUDE_REPLIES = True
MAX_COMMENTS_PER_VIDEO = 250
MAX_VIDEOS_PER_CHANNEL = 5
REQUEST_TIMEOUT = 30
RUN_COLLECTION = True

SEED_VIDEOS = [
    {
        "video_id": "LGDV0DW_Y1g",
        "seed_type": "video_id",
        "seed_value": "LGDV0DW_Y1g",
        "seed_reason": "manual seed from user-selected YouTube video for judol dataset pilot",
        "paper_basis": "seed-video collection strategy from YouTube judol papers",
        "collection_query": "manual_seed:https://youtu.be/LGDV0DW_Y1g",
    },
    {
        "video_id": "c7IYpLO44hc",
        "seed_type": "video_id",
        "seed_value": "c7IYpLO44hc",
        "seed_reason": "manual seed from user-selected YouTube video for judol dataset pilot",
        "paper_basis": "seed-video collection strategy from YouTube judol papers",
        "collection_query": "manual_seed:https://youtu.be/c7IYpLO44hc",
    },
    {
        "video_id": "-zKVlGqDld8",
        "seed_type": "video_id",
        "seed_value": "-zKVlGqDld8",
        "seed_reason": "manual seed from user-selected YouTube live video for judol dataset pilot",
        "paper_basis": "seed-video collection strategy from YouTube judol papers",
        "collection_query": "manual_seed:https://www.youtube.com/live/-zKVlGqDld8",
    },
    {
        "video_id": "PIan6qn8FwE",
        "seed_type": "video_id",
        "seed_value": "PIan6qn8FwE",
        "seed_reason": "manual seed from user-selected YouTube video for judol dataset pilot",
        "paper_basis": "seed-video collection strategy from YouTube judol papers",
        "collection_query": "manual_seed:https://youtu.be/PIan6qn8FwE",
    },
    {
        "video_id": "dw3-zZyvJXg",
        "seed_type": "video_id",
        "seed_value": "dw3-zZyvJXg",
        "seed_reason": "manual seed from user-selected YouTube video for judol dataset pilot",
        "paper_basis": "seed-video collection strategy from YouTube judol papers",
        "collection_query": "manual_seed:https://youtu.be/dw3-zZyvJXg",
    },
]

SEED_CHANNELS = []

BASELINE_KEYWORDS = [
    "slot", "gacor", "judi", "togel", "bonus", "deposit", "depo",
    "maxwin", "daftar", "link", "jackpot", "freebet", "cuan", "untung",
]

CONTACT_PATTERNS = [
    r"https?://",
    r"www\.",
    r"t\.me/",
    r"telegram",
    r"whatsapp",
    r"\bwa\b",
    r"line\b",
    r"@\w{4,}",
]

print({
    "project_root": str(PROJECT_ROOT),
    "env_path": str(ENV_PATH),
    "env_file_present": ENV_PATH.exists(),
    "api_key_source": API_KEY_SOURCE,
    "api_key_length": len(YOUTUBE_API_KEY),
    "loaded_env_keys": sorted(LOADED_ENV.keys()),
    "output_dir": str(OUTPUT_DIR),
    "seed_videos": len(SEED_VIDEOS),
    "seed_channels": len(SEED_CHANNELS),
    "max_comments_per_video": MAX_COMMENTS_PER_VIDEO,
    "include_replies": INCLUDE_REPLIES,
    "export_format": EXPORT_FORMAT,
    "api_key_present": bool(YOUTUBE_API_KEY),
})


## Helper Functions

Pisahkan helper API, normalization, enrichment, dan export agar notebook tetap mudah diaudit.


In [ ]:
API_BASE_URL = "https://www.googleapis.com/youtube/v3"
ASCII_WORD_RE = re.compile(r"[a-z0-9]+")
URL_OR_CONTACT_RE = re.compile("|".join(CONTACT_PATTERNS), re.IGNORECASE)
SPACED_WORD_RE = re.compile(r"(?:[a-zA-Z0-9]\s+){2,}[a-zA-Z0-9]")
ZERO_WIDTH_CHARS = {"\u200b", "\u200c", "\u200d", "\ufeff"}


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def normalize_text(text: str) -> str:
    if not text:
        return ""
    lowered = text.casefold()
    normalized = unicodedata.normalize("NFKC", lowered)
    collapsed = re.sub(r"\s+", " ", normalized).strip()
    return collapsed


def compact_alnum_text(text: str) -> str:
    pieces = ASCII_WORD_RE.findall(normalize_text(text))
    return "".join(pieces)


def detect_keyword_matches(text: str, keywords: Iterable[str]) -> list[str]:
    normalized = normalize_text(text)
    compact = compact_alnum_text(text)
    matches = []
    for keyword in keywords:
        key_norm = normalize_text(keyword)
        key_compact = re.sub(r"[^a-z0-9]+", "", key_norm)
        if key_norm and key_norm in normalized:
            matches.append(keyword)
            continue
        if key_compact and key_compact in compact:
            matches.append(keyword)
    return sorted(set(matches))


def detect_contact_flag(text: str) -> bool:
    return bool(URL_OR_CONTACT_RE.search(text or ""))


def detect_unicode_flags(text: str) -> list[str]:
    if not text:
        return []
    flags = []
    if any(ord(ch) > 127 for ch in text):
        flags.append("non_ascii")
    if any(ch in ZERO_WIDTH_CHARS for ch in text):
        flags.append("zero_width")
    if SPACED_WORD_RE.search(text):
        flags.append("irregular_spacing")
    if re.search(r"[a-zA-Z][0-9][a-zA-Z]|[0-9][a-zA-Z]|[a-zA-Z][0-9]", text):
        flags.append("number_substitution")
    if re.search(r"[a-zA-Z][^\w\s][a-zA-Z]", text):
        flags.append("symbol_splitting")
    return sorted(set(flags))


def guess_language_hint(text: str) -> str:
    sample = normalize_text(text)
    id_markers = ["yang", "dan", "untuk", "dengan", "ini", "itu", "cara", "bonus"]
    score = sum(1 for token in id_markers if token in sample)
    if score >= 2:
        return "id_likely"
    if sample:
        return "unknown"
    return "empty"


def hash_author_name(author_name: str) -> str:
    value = (author_name or "").strip()
    if not value:
        return ""
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def enrich_comment_row(row: dict, keywords: Iterable[str]) -> dict:
    text = row.get("text_original", "")
    keyword_matches = detect_keyword_matches(text, keywords)
    unicode_flags = detect_unicode_flags(text)

    enriched = dict(row)
    enriched["normalized_text"] = normalize_text(text)
    enriched["keyword_matches"] = keyword_matches
    enriched["candidate_flag_keyword"] = bool(keyword_matches)
    enriched["candidate_flag_contact"] = detect_contact_flag(text)
    enriched["candidate_flag_unicode"] = bool(unicode_flags)
    enriched["unicode_obfuscation_flags"] = unicode_flags
    enriched["language_hint"] = guess_language_hint(text)
    enriched.setdefault("platform", "youtube")
    enriched.setdefault("source_type", "api")
    enriched.setdefault("label", "perlu_review")
    enriched.setdefault("label_source", "unlabeled")
    enriched.setdefault("notes", "")
    return enriched


def build_annotation_queue(df_comments: pd.DataFrame) -> pd.DataFrame:
    if df_comments.empty:
        return df_comments.copy()
    ranked = df_comments.copy()
    ranked["candidate_score"] = (
        ranked["candidate_flag_keyword"].astype(int)
        + ranked["candidate_flag_contact"].astype(int)
        + ranked["candidate_flag_unicode"].astype(int)
    )
    ranked = ranked.sort_values(
        ["candidate_score", "published_at"],
        ascending=[False, True],
        na_position="last",
    )
    return ranked


In [ ]:
def youtube_api_get(endpoint: str, params: dict) -> dict:
    if not YOUTUBE_API_KEY:
        raise RuntimeError("YOUTUBE_API_KEY is empty. Set it before running collection.")

    clean_params = {
        key: value
        for key, value in params.items()
        if value is not None and value != ""
    }
    query_string = urlencode({**clean_params, "key": YOUTUBE_API_KEY})
    url = f"{API_BASE_URL}/{endpoint}?{query_string}"

    try:
        with urlopen(url, timeout=REQUEST_TIMEOUT) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as error:
        body = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"YouTube API {endpoint} failed with HTTP {error.code}: {body}"
        ) from error
    except URLError as error:
        raise RuntimeError(f"YouTube API {endpoint} request failed: {error}") from error

    if "error" in payload:
        raise RuntimeError(json.dumps(payload["error"], ensure_ascii=False))
    return payload


def fetch_video_details(video_ids: list[str]) -> list[dict]:
    if not video_ids:
        return []
    payload = youtube_api_get(
        "videos",
        {
            "part": "snippet,statistics,contentDetails",
            "id": ",".join(video_ids),
            "maxResults": len(video_ids),
        },
    )
    rows = []
    collected_at = utc_now_iso()
    for item in payload.get("items", []):
        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        rows.append({
            "video_id": item.get("id", ""),
            "channel_id": snippet.get("channelId", ""),
            "channel_title": snippet.get("channelTitle", ""),
            "video_title": snippet.get("title", ""),
            "video_description": snippet.get("description", ""),
            "published_at": snippet.get("publishedAt", ""),
            "category_hint": snippet.get("categoryId", ""),
            "view_count": statistics.get("viewCount"),
            "like_count": statistics.get("likeCount"),
            "comment_count": statistics.get("commentCount"),
            "duration": item.get("contentDetails", {}).get("duration", ""),
            "collected_at": collected_at,
        })
    return rows


def fetch_channel_video_ids(channel_id: str, max_videos: int) -> list[str]:
    next_page_token = None
    video_ids = []

    while len(video_ids) < max_videos:
        payload = youtube_api_get(
            "search",
            {
                "part": "snippet",
                "channelId": channel_id,
                "type": "video",
                "order": "date",
                "maxResults": min(50, max_videos - len(video_ids)),
                "pageToken": next_page_token,
            },
        )
        for item in payload.get("items", []):
            video_id = item.get("id", {}).get("videoId")
            if video_id and video_id not in video_ids:
                video_ids.append(video_id)
        next_page_token = payload.get("nextPageToken")
        if not next_page_token:
            break

    return video_ids


def fetch_comment_rows_for_video(video_seed: dict, max_comments: int, include_replies: bool) -> list[dict]:
    video_id = video_seed["video_id"]
    rows = []
    next_page_token = None

    while len(rows) < max_comments:
        payload = youtube_api_get(
            "commentThreads",
            {
                "part": "snippet,replies",
                "videoId": video_id,
                "maxResults": min(100, max_comments - len(rows)),
                "order": "time",
                "pageToken": next_page_token,
                "textFormat": "plainText",
            },
        )

        for item in payload.get("items", []):
            top_level = item.get("snippet", {}).get("topLevelComment", {})
            top_snippet = top_level.get("snippet", {})
            base_row = {
                "platform": "youtube",
                "source_type": "api",
                "source_seed": video_seed.get("seed_value", video_id),
                "collection_query": video_seed.get("collection_query", f"video_id:{video_id}"),
                "paper_basis": video_seed.get("paper_basis", ""),
                "video_id": video_id,
                "comment_id": top_level.get("id", ""),
                "parent_comment_id": "",
                "author_name_hash": hash_author_name(top_snippet.get("authorDisplayName", "")),
                "text_original": top_snippet.get("textDisplay", ""),
                "published_at": top_snippet.get("publishedAt", ""),
                "updated_at": top_snippet.get("updatedAt", ""),
                "like_count": top_snippet.get("likeCount"),
                "reply_count": item.get("snippet", {}).get("totalReplyCount", 0),
                "is_reply": False,
                "collected_at": utc_now_iso(),
                "label": "perlu_review",
                "label_source": "unlabeled",
                "notes": "",
            }
            rows.append(base_row)

            if include_replies:
                for reply in item.get("replies", {}).get("comments", []):
                    reply_snippet = reply.get("snippet", {})
                    rows.append({
                        "platform": "youtube",
                        "source_type": "api",
                        "source_seed": video_seed.get("seed_value", video_id),
                        "collection_query": video_seed.get("collection_query", f"video_id:{video_id}"),
                        "paper_basis": video_seed.get("paper_basis", ""),
                        "video_id": video_id,
                        "comment_id": reply.get("id", ""),
                        "parent_comment_id": top_level.get("id", ""),
                        "author_name_hash": hash_author_name(reply_snippet.get("authorDisplayName", "")),
                        "text_original": reply_snippet.get("textDisplay", ""),
                        "published_at": reply_snippet.get("publishedAt", ""),
                        "updated_at": reply_snippet.get("updatedAt", ""),
                        "like_count": reply_snippet.get("likeCount"),
                        "reply_count": 0,
                        "is_reply": True,
                        "collected_at": utc_now_iso(),
                        "label": "perlu_review",
                        "label_source": "unlabeled",
                        "notes": "",
                    })

            if len(rows) >= max_comments:
                break

        next_page_token = payload.get("nextPageToken")
        if not next_page_token or len(rows) >= max_comments:
            break

    deduped = {row["comment_id"]: row for row in rows if row.get("comment_id")}
    return list(deduped.values())


## Offline Smoke Check

Cell ini bisa dijalankan tanpa API key untuk memverifikasi enrichment minimum sebelum ambil data asli.


In [ ]:
sample_rows = [
    {"text_original": "bonus new member slot gacor deposit 10k"},
    {"text_original": "slot waktu praktikum sudah penuh"},
    {"text_original": "b0nus s l o t gacor via t.me/demo"},
]

smoke_df = pd.DataFrame([enrich_comment_row(row, BASELINE_KEYWORDS) for row in sample_rows])
assert bool(smoke_df.loc[0, "candidate_flag_keyword"])
assert smoke_df.loc[1, "label"] == "perlu_review"
assert bool(smoke_df.loc[2, "candidate_flag_contact"])
assert bool(smoke_df.loc[2, "candidate_flag_unicode"])
smoke_df[["text_original", "keyword_matches", "candidate_flag_contact", "unicode_obfuscation_flags"]]


## Collect Seed Videos

Notebook akan menggabungkan `SEED_VIDEOS` eksplisit dengan video yang diturunkan dari `SEED_CHANNELS`. Provenance untuk setiap seed tetap dipertahankan.


In [ ]:
resolved_video_seeds = []
seen_video_ids = set()

for seed in SEED_VIDEOS:
    video_id = seed.get("video_id", "").strip()
    if video_id and video_id not in seen_video_ids:
        resolved_video_seeds.append(dict(seed))
        seen_video_ids.add(video_id)

if RUN_COLLECTION and YOUTUBE_API_KEY:
    for channel_seed in SEED_CHANNELS:
        channel_video_ids = fetch_channel_video_ids(
            channel_seed["channel_id"],
            max_videos=MAX_VIDEOS_PER_CHANNEL,
        )
        for video_id in channel_video_ids:
            if video_id in seen_video_ids:
                continue
            resolved_video_seeds.append({
                "video_id": video_id,
                "seed_type": channel_seed["seed_type"],
                "seed_value": channel_seed["seed_value"],
                "seed_reason": channel_seed["seed_reason"],
                "paper_basis": channel_seed["paper_basis"],
                "collection_query": channel_seed["collection_query"],
            })
            seen_video_ids.add(video_id)
else:
    print("Skipping channel expansion because API key is missing or RUN_COLLECTION is False.")

resolved_video_seeds[:5]


## Fetch Video Metadata and Comments

Jika API key tersedia, cell ini akan memanggil endpoint resmi untuk metadata video, top-level comments, dan replies. Jika tidak, notebook tetap runnable dan akan melewati tahap koleksi.


In [ ]:
video_rows = []
comment_rows = []
collection_errors = []

if RUN_COLLECTION and YOUTUBE_API_KEY and resolved_video_seeds:
    requested_video_ids = [seed["video_id"] for seed in resolved_video_seeds]
    details_lookup = {
        row["video_id"]: row
        for row in fetch_video_details(requested_video_ids)
    }

    for seed in resolved_video_seeds:
        video_id = seed["video_id"]
        video_metadata = details_lookup.get(video_id)
        if not video_metadata:
            collection_errors.append({
                "video_id": video_id,
                "stage": "video_details",
                "reason": "video_not_returned_by_api",
                "seed_reason": seed.get("seed_reason", ""),
            })
            continue

        metadata = dict(video_metadata)
        metadata.update({
            "seed_type": seed.get("seed_type", "video_id"),
            "seed_value": seed.get("seed_value", video_id),
            "seed_reason": seed.get("seed_reason", ""),
            "paper_basis": seed.get("paper_basis", ""),
            "collection_query": seed.get("collection_query", f"video_id:{video_id}"),
            "platform": "youtube",
            "source_type": "api",
        })
        video_rows.append(metadata)

        try:
            fetched_comments = fetch_comment_rows_for_video(
                seed,
                max_comments=MAX_COMMENTS_PER_VIDEO,
                include_replies=INCLUDE_REPLIES,
            )
            comment_rows.extend(fetched_comments)
        except RuntimeError as error:
            collection_errors.append({
                "video_id": video_id,
                "stage": "comment_threads",
                "reason": str(error),
                "seed_reason": seed.get("seed_reason", ""),
            })
else:
    print("Collection skipped. Provide YOUTUBE_API_KEY and keep RUN_COLLECTION=True to fetch live data.")

df_videos = pd.DataFrame(video_rows)
df_comments_raw = pd.DataFrame(comment_rows)
df_collection_errors = pd.DataFrame(collection_errors)

print({
    "videos_requested": len(resolved_video_seeds),
    "videos_available": len(df_videos),
    "comments_raw": len(df_comments_raw),
    "collection_errors": len(df_collection_errors),
})

if not df_collection_errors.empty:
    display(df_collection_errors)


## Enrich and Prepare Annotation Queue

Tambahkan candidate flags, dedup ringan, dan kolom stabil untuk downstream anotasi.


In [ ]:
if df_comments_raw.empty:
    df_comments = df_comments_raw.copy()
    df_annotation = df_comments_raw.copy()
else:
    enriched_rows = [enrich_comment_row(row, BASELINE_KEYWORDS) for row in df_comments_raw.to_dict(orient="records")]
    df_comments = pd.DataFrame(enriched_rows)
    df_comments = df_comments.drop_duplicates(subset=["comment_id"], keep="first")
    df_annotation = build_annotation_queue(df_comments)

annotation_columns = [
    "platform",
    "source_type",
    "source_seed",
    "collection_query",
    "paper_basis",
    "video_id",
    "comment_id",
    "parent_comment_id",
    "text_original",
    "normalized_text",
    "published_at",
    "like_count",
    "reply_count",
    "is_reply",
    "keyword_matches",
    "candidate_flag_keyword",
    "candidate_flag_contact",
    "candidate_flag_unicode",
    "unicode_obfuscation_flags",
    "language_hint",
    "label",
    "label_source",
    "notes",
    "collected_at",
]

df_annotation = df_annotation.reindex(columns=annotation_columns)
df_annotation.head(10)


## Export

Notebook menghasilkan dua keluaran: dataset mentah dan queue anotasi. `csv` adalah default paling aman untuk kerja awal.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_path = OUTPUT_DIR / f"{RAW_OUTPUT_STEM}.{EXPORT_FORMAT}"
annotation_path = OUTPUT_DIR / f"{ANNOTATION_OUTPUT_STEM}.{EXPORT_FORMAT}"

if EXPORT_FORMAT == "csv":
    if not df_comments.empty:
        df_comments.to_csv(raw_path, index=False)
        df_annotation.to_csv(annotation_path, index=False)
elif EXPORT_FORMAT == "parquet":
    if not df_comments.empty:
        df_comments.to_parquet(raw_path, index=False)
        df_annotation.to_parquet(annotation_path, index=False)
else:
    raise ValueError(f"Unsupported EXPORT_FORMAT: {EXPORT_FORMAT}")

print({
    "raw_path": str(raw_path),
    "annotation_path": str(annotation_path),
    "raw_rows": int(len(df_comments)),
    "annotation_rows": int(len(df_annotation)),
})


## Results

Audit kecil di bawah ini membantu memeriksa provenance seed, distribusi candidate flag, dan komposisi reply.


In [ ]:
if df_comments.empty:
    print("No comments collected yet.")
else:
    audit = {
        "total_videos": int(df_videos["video_id"].nunique()) if not df_videos.empty else 0,
        "total_comments": int(len(df_comments)),
        "total_replies": int(df_comments["is_reply"].sum()),
        "keyword_candidates": int(df_comments["candidate_flag_keyword"].sum()),
        "contact_candidates": int(df_comments["candidate_flag_contact"].sum()),
        "unicode_candidates": int(df_comments["candidate_flag_unicode"].sum()),
        "seed_sources": df_videos["seed_type"].value_counts(dropna=False).to_dict() if not df_videos.empty else {},
    }
    print(audit)

    display(df_videos[["video_id", "video_title", "seed_type", "seed_reason", "paper_basis"]].head(10))
    display(df_comments[["comment_id", "video_id", "candidate_flag_keyword", "candidate_flag_contact", "candidate_flag_unicode", "text_original"]].head(10))


## Takeaways

- Jalur koleksi utama sekarang adalah **YouTube Data API v3**, bukan HTML scraping atau keyword-only video search.
- Seed provenance disimpan sejak awal agar metodologi bisa dijelaskan ke dosen.
- Keyword dipakai sebagai evidence untuk prioritas anotasi, bukan label otomatis.
- Unicode suspicion, contact evidence, dan replies sudah masuk ke artefak pilot sehingga cocok untuk langkah anotasi berikutnya.
